# Stepping through peak filtering

A small signal with two cycles, each with a jagged top. Step 1 finds every local peak above a threshold with masks. Step 2 walks through those candidates one at a time and keeps one peak per cluster, the same loop as `find_peaks` in the Features from data notebook. The figure at the end steps through both, one click at a time.

In [17]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

## The data

Two cycles sampled every 0.05 s. The tops are jagged, and there is a small wiggle between the cycles.

In [18]:
t = np.round(0.05 * np.arange(24), 2)
a = np.array([0.00, 0.20, 0.45, 0.70, 0.65, 0.90, 1.00, 0.85, 0.90, 0.60, 0.20, 0.30,
              0.05, 0.35, 0.70, 0.95, 0.80, 0.85, 0.55, 0.30, 0.10, 0.15, 0.05, 0.00])

threshold = 0.5 * a.max()
min_spacing = 0.25

## Step 1: every local peak above the threshold

A sample is a local peak when it is larger than the sample before it and the sample after it. Each condition is a mask built from a slice of `a`, and `&` combines them. A third mask keeps only the peaks above the threshold. The masks start at the second sample, so the indices need `+ 1`.

In [19]:
local_peak = (
    (a[1:-1] > a[:-2])            # larger than the previous point
    & (a[1:-1] > a[2:]))          # larger than the next point
is_peak = local_peak & (a[1:-1] > threshold)   # and larger than the threshold

local_index = np.where(local_peak)[0] + 1
candidates = np.where(is_peak)[0] + 1
print("local peaks:", local_index)
print("candidates: ", candidates)

local peaks: [ 3  6  8 11 15 17 21]
candidates:  [ 3  6  8 15 17]


## Step 2: one peak per cluster

This is the body of the loop, written as a function so the figure below can call it one candidate at a time. A candidate closer than `min_spacing` to the last kept peak belongs to the same peak: it replaces that peak if it is taller, and is dropped otherwise.

In [20]:
def consider(i, keep, dropped):
    """One pass of the loop: decide candidate i, update keep and dropped, return the reason."""
    if len(keep) == 0:
        keep.append(i)
        return f"index {i}: nothing kept yet, so keep it"
    gap = t[i] - t[keep[-1]]
    if gap > min_spacing:
        keep.append(i)
        return f"index {i}: gap {gap:.2f} s > {min_spacing} s, so it is a new peak; keep it"
    if a[i] > a[keep[-1]]:
        dropped.append(keep[-1])
        old, keep[-1] = keep[-1], i
        return (f"index {i}: gap {gap:.2f} s ≤ {min_spacing} s, same peak as {old}, "
                f"but taller ({a[i]:.2f} > {a[old]:.2f}); it replaces {old}")
    dropped.append(i)
    return (f"index {i}: gap {gap:.2f} s ≤ {min_spacing} s, same peak as {keep[-1]}, "
            f"and not taller ({a[i]:.2f} ≤ {a[keep[-1]]:.2f}); drop it")

## Step through it

Run the cell, then use the buttons. **Next** and **Back** move one stage, **Reset** returns to the raw data. The first stages build the masks from Step 1; after that, each click considers one candidate. The black ring marks it, and the arrow is the gap back to the last kept peak. The green band is `min_spacing` measured back from the candidate: if the last kept peak is inside the band, the candidate belongs to the same peak.

In [21]:
INTRO = [
    "the signal: two cycles with jagged tops",
    f"threshold = 0.5 × max = {threshold:.2f}",
    f"local peaks, larger than both neighbours: {local_index.tolist()}",
    f"candidates, local peaks above the threshold too: {candidates.tolist()}",
]
n_stages = len(INTRO) + len(candidates) + 1          # intro, one per candidate, result

with plt.ioff():
    fig, ax = plt.subplots(figsize=(9, 4.2))
fig.canvas.header_visible = False


def draw(stage):
    """Draw the figure for one stage by replaying the steps from the start."""
    ax.cla()
    ax.plot(t, a, "-o", ms=4, lw=1, color="C0", zorder=1)
    ax.set_ylim(-0.05, 1.32)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("acceleration (g)")
    if stage == 0:
        for j in range(len(a)):
            ax.annotate(str(j), (t[j], a[j]), textcoords="offset points", xytext=(0, 7),
                        ha="center", fontsize=7, color="gray")
    if stage >= 1:
        ax.axhline(threshold, color="gray", linestyle=":", label="threshold")
    if stage == 2:
        ax.plot(t[local_index], a[local_index], "o", ms=9, mfc="none", mec="C1", mew=2,
                label="local peaks")
    if stage >= 3:
        below = np.setdiff1d(local_index, candidates)
        ax.plot(t[below], a[below], "o", ms=9, mfc="none", mec="lightgray", mew=2,
                label="below threshold" if stage == 3 else None)
        ax.plot(t[candidates], a[candidates], "o", ms=9, mfc="none", mec="C1", mew=2,
                label="candidates")
    if stage >= 2:
        shown = local_index if stage == 2 else candidates
        for j in shown:
            ax.annotate(str(j), (t[j], a[j]), textcoords="offset points", xytext=(9, 0),
                        va="center", fontsize=8, color="C1")
    if stage < len(INTRO):
        message = INTRO[stage]
    else:
        keep, dropped = [], []
        n_done = min(stage - len(INTRO) + 1, len(candidates))
        for i in candidates[:n_done]:
            i = int(i)
            last = keep[-1] if keep else None
            message = consider(i, keep, dropped)
        ax.plot(t[keep], a[keep], "o", ms=9, color="C2", label="kept")
        ax.plot(t[dropped], a[dropped], "x", ms=10, mew=2.5, color="C3", label="dropped")
        if stage < n_stages - 1:
            ax.plot([t[i]], [a[i]], "o", ms=20, mfc="none", mec="k", mew=2)
            if last is not None:
                # the ruler: min_spacing measured back from the candidate
                ax.axvspan(t[i] - min_spacing, t[i], color="C2", alpha=0.12)
                y = 1.2
                ax.annotate("", xy=(t[i], y), xytext=(t[last], y),
                            arrowprops=dict(arrowstyle="<->", color="k"))
                ax.text((t[i] + t[last]) / 2, y + 0.02, f"gap {t[i] - t[last]:.2f} s",
                        ha="center", va="bottom", fontsize=8)
        else:
            spacing = np.diff(t[keep])
            message = f"keep = {keep}; spacing between the peaks {spacing.round(2).tolist()} s"
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title(f"{stage + 1}/{n_stages}   {message}", fontsize=10)
    fig.canvas.draw_idle()


stage = 0
back = widgets.Button(description="Back", icon="arrow-left")
step = widgets.Button(description="Next", icon="arrow-right", button_style="primary")
reset = widgets.Button(description="Reset", icon="refresh")


def go(new_stage):
    global stage
    stage = max(0, min(new_stage, n_stages - 1))
    back.disabled = stage == 0
    step.disabled = stage == n_stages - 1
    draw(stage)


back.on_click(lambda _: go(stage - 1))
step.on_click(lambda _: go(stage + 1))
reset.on_click(lambda _: go(0))
go(0)
display(widgets.VBox([widgets.HBox([reset, back, step]), fig.canvas]))

/var/folders/q0/kxmqm5c95n7cxc6mmqklw1k40000gq/T/ipykernel_59918/2021896851.py:65: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="upper right", fontsize=8)
